# Congested Reward Safety Factor Study

Sweep a scalar `reward_safety_factor` for the Attention DQN + Adaptive TTC + TTC-observation setup. The factor interpolates between aggressive flow/reward seeking and stronger TTC/collision safety shaping.

The baseline cell below runs plain DQN in congested traffic where surrounding cars receive different driver-aggressiveness profiles, and the ego observation includes a normalized `driver_aggressiveness` feature for each observed vehicle.


In [ ]:
from pathlib import Path
import importlib
import sys

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root.")

PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "congested_traffic_policy"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import reward_safety_factor_study as study
study = importlib.reload(study)

print("Study results dir:", study.RESULTS_DIR)
print("Safety factors:", study.SAFETY_FACTORS)
print("Timesteps per run:", study.TIMESTEPS)


## Inspect Reward Configs


In [ ]:
import pandas as pd

rows = []
for factor in study.SAFETY_FACTORS:
    reward = study.make_reward_config(factor)
    safety = study.make_safety_ttc_flow_reward_config(factor)
    rows.append({
        "reward_safety_factor": factor,
        "collision_reward": reward["collision_reward"],
        "high_speed_reward": reward["high_speed_reward"],
        "lane_change_reward": reward["lane_change_reward"],
        "safety_enabled": safety.get("enabled", False),
        "low_ttc_penalty_weight": safety.get("low_ttc_penalty_weight", 0.0),
        "max_low_ttc_penalty": safety.get("max_low_ttc_penalty", 0.0),
        "lag_penalty_weight": safety.get("lag_penalty_weight", 0.0),
        "speed_tolerance": safety.get("speed_tolerance", None),
    })
pd.DataFrame(rows)


## Run Baseline DQN + Aggressiveness State


In [ ]:
# Plain baseline DQN. Traffic vehicles have mixed aggressiveness, and the normalized
# driver_aggressiveness index is appended to each observed vehicle row in the ego state.
# For a quick smoke run, use: study.run_baseline_aggressiveness_state(timesteps=5_000)
baseline_aggressiveness_output = study.run_baseline_aggressiveness_state(timesteps=study.TIMESTEPS)
baseline_aggressiveness_output["saved_eval_summary"]["metric_summary"]


## Load Existing Comparison


In [ ]:
comparison_path = study.RESULTS_DIR / "reward_safety_factor_comparison.csv"
if comparison_path.exists():
    loaded_comparison_df = pd.read_csv(comparison_path)
    display(loaded_comparison_df.round(3))
else:
    print("No comparison file yet:", comparison_path)


## Tradeoff Plot


In [ ]:
import matplotlib.pyplot as plt

plot_df = globals().get("comparison_df", globals().get("loaded_comparison_df", None))
if plot_df is None or plot_df.empty:
    print("Run or load the study first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    x = plot_df["reward_safety_factor"]
    axes[0].plot(x, plot_df.get("collision_rate_%_mean"), marker="o", color="crimson")
    axes[0].set_title("Collision Rate")
    axes[0].set_ylabel("%")
    axes[1].plot(x, plot_df.get("overtakes_mean"), marker="o", color="tab:orange")
    axes[1].set_title("Overtakes")
    axes[2].plot(x, plot_df.get("reward_mean"), marker="o", color="tab:green")
    axes[2].set_title("Reward")
    for ax in axes:
        ax.set_xlabel("reward_safety_factor")
        ax.grid(alpha=0.3)
    fig.tight_layout()
    plot_path = study.RESULTS_DIR / "reward_safety_factor_tradeoff.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    print("Saved plot:", plot_path)
    plt.show()
